# Module 4: Dense versus MoE on the GPU

In Module 3 you proved one request leaves the GPU memory-bound: to make one token, the server reads every weight once. This module gives that fact a picture and a name, then uses it to explain why a Mixture-of-Experts model generates faster than its size suggests. You plot the compute limit and the memory limit, find where they cross, and place decode far down the memory side. Then you derive, from the model configs, why an MoE moves fewer bytes per token than its total size. The plot runs live. The MoE comparison ships as commented-out code, because a 30B model does not fit the workshop card.

## Learning objectives
- Plot compute against memory bandwidth and read the ridge point for your card
- Place decode and prefill on that plot and say which is memory-bound
- Explain why decode reads every weight to make one token
- Derive active versus total parameters for a Mixture-of-Experts model and the bytes each moves per token
- Name the five ways to read fewer bytes, and which module covers each

- The live comparison (section 5) switches models with `vllm_admin.switch_model`, which restarts your vLLM (seconds, from the PVC cache)

References: [vLLM](https://docs.vllm.ai) &middot; [Anatomy of vLLM](https://blog.vllm.ai/2025/09/05/anatomy-of-vllm.html) &middot; [Qwen3-30B-A3B model card](https://huggingface.co/Qwen/Qwen3-30B-A3B) &middot; [RTX 4000 Ada datasheet](https://www.nvidia.com/en-us/products/workstations/rtx-4000/)

## Memory-bound design basics

Decode is memory-bound. To make one token, the server reads every weight in the model once, so the token rate is capped by how fast the card reads memory. Every technique in Part 2 is a way to read fewer bytes or reuse the bytes you read.

- Arithmetic intensity is the work done per byte read, in FLOPs per byte. You computed it in Module 3.
- Plot two limits against it: a sloped memory-bandwidth line and a flat compute line. Where they cross is the ridge point. The plot is the roofline.
- Decode at batch 1 sits far down the memory side. Prefill sits near the compute line. Same card, two different places.

![The memory-bandwidth line and the compute line meeting at the ridge point, with decode at batch 1 far down the memory side and prefill near the top](images/04_dense_vs_moe_architecture.png)

## 1. Setup

The plot and the MoE math need only numpy and matplotlib. The dense measurement in section 5 uses your settings and client. We reinstall here so this notebook stands on its own.

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
# Imports, and your settings for the one live measurement later.
import os, sys, time
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
from common.config import get_settings, build_client

settings = get_settings()
print("model   :", settings.model_name)
print("endpoint:", settings.vllm_host)

## 2. Memory-bound versus compute-bound, by hand (no GPU)

Plot the two limits. The compute line is flat at the card's peak FLOPs. The memory line slopes up with arithmetic intensity, at the card's bandwidth. Below the crossover you are memory-bound, above it compute-bound. Use the RTX 4000 Ada numbers: about 107 TFLOP/s dense FP16 and 360 GB/s.

In [ ]:
# The roofline: compute ceiling, memory ceiling, and where they cross (the ridge point).
peak_tflops = 107.0     # RTX 4000 Ada, dense FP16. NOT 150: 427 on the sheet is FP8 with 2:1 sparsity.
bandwidth_tbs = 0.36    # 360 GB/s

ridge = peak_tflops / bandwidth_tbs   # FLOPs per byte where memory-bound flips to compute-bound
print(f"ridge point: {ridge:.0f} FLOPs/byte  (below this you are memory-bound)")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

intensity = np.logspace(-1, 3, 200)                              # FLOPs per byte, log scale
attainable = np.minimum(peak_tflops, bandwidth_tbs * intensity)  # TFLOP/s actually reachable

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(intensity, attainable)
ax.axvline(ridge, ls="--", color="grey")
ax.scatter([1.0], [bandwidth_tbs * 1.0], color="tab:red", zorder=5, label="decode at batch 1")
ax.scatter([ridge * 3], [peak_tflops], color="tab:green", zorder=5, label="prefill")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("arithmetic intensity (FLOPs/byte)")
ax.set_ylabel("attainable TFLOP/s")
ax.set_title(f"Memory-bound below {ridge:.0f} FLOPs/byte, compute-bound above")
ax.legend(); ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()

print(f"decode at intensity ~1 reaches {bandwidth_tbs * 1.0:.2f} TFLOP/s, "
      f"about {bandwidth_tbs * 1.0 / peak_tflops * 100:.2f}% of peak")

**What you should see:** a ridge near 297 FLOPs per byte, a red dot for decode sitting far down the sloped memory side, and a tiny fraction of peak compute reached. The tensor cores wait on memory. You measured this split in Module 3; this is the picture of it.

## 3. Why decode is memory-bound, in one sentence

To make one token, the server reads every weight in the model once, so your single-stream token rate is capped by how fast the card reads memory, not by how much math it can do. Every technique in Part 2 chips at that one limit.

## 4. MoE: the first way to read fewer weights (derive from the configs)

A dense model reads all its weights per token. A Mixture-of-Experts model holds many experts but routes each token to only a few, so it reads only the active ones. Total parameters set the memory footprint. Active parameters set the bandwidth cost, and bandwidth caps decode. Read the two from the configs and compare the bytes each moves per token.

In [ ]:
# Bytes moved per token at FP16. The MoE wins by reading only its active experts.
bpp = 2

dense4_read  = 4.0e9  * bpp     # dense 4B reads all 4B
dense30_read = 30.5e9 * bpp     # a dense 30B would read all 30.5B (hypothetical, for contrast)
moe_read     = 3.3e9  * bpp     # Qwen3-30B-A3B reads only ~3.3B active params (8 of 128 experts)
moe_store    = 30.5e9 * bpp     # but stores all 128 experts

print(f"dense 4B    : {dense4_read/1e9:5.1f} GB read per token")
print(f"dense 30B   : {dense30_read/1e9:5.1f} GB read per token  (hypothetical, for contrast)")
print(f"MoE 30B-A3B : {moe_read/1e9:5.1f} GB read per token, but {moe_store/1e9:.0f} GB stored")
print(f"=> the MoE reads {dense30_read/moe_read:.1f}x fewer bytes than a dense 30B, "
      f"so it decodes about that much faster")

**What you should see:** the MoE reads about 6.6 GB per token, roughly nine times less than a dense 30B, so it decodes about nine times faster while answering with 30B-scale quality. The catch is the footprint: all 128 experts sit in VRAM, about 61 GB at FP16, 30 GB at FP8, and 15 to 18 GB at INT4. It fits the 20 GB card only at INT4 and a small context, which is why the next section is a talk-through.

## 5. Compare two dense models, live

You have three models pre-cached, a 4B and two 0.6B. Switching restarts your vLLM and loads from the PVC cache in seconds. Measure the decode rate on the model you are serving, switch to a 0.6B, and measure again. The smaller model reads fewer weight bytes per token, so it decodes faster. That is the memory-bound limit, measured on your own card. The Mixture-of-Experts model is the clever middle, 30B-scale quality at close to a small model's per-token read, but it does not fit this card, so it stays the commented reference below.

In [ ]:
# A small decode-rate probe: stream one answer and time the tokens.
from common.config import build_client
client = build_client(settings)

def decode_rate(model):
    start = time.time(); first = None; n = 0
    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": "Write a detailed paragraph about GPU memory."}],
        max_tokens=200, temperature=0.0, stream=True,
    )
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            if first is None:
                first = time.time()
            n += 1
    secs = (time.time() - first) if first else 0.0
    return n / secs if secs else 0.0

print(f"{settings.model_name}: about {decode_rate(settings.model_name):.0f} tokens/s")

**What you should see:** a decode rate near the bandwidth-over-weights ceiling from Module 2, the memory-bound floor for one request. The next cell switches to the 0.6B and measures again. You should see it decode several times faster, on the same card. The only change is how many weight bytes each token reads, which is the whole lesson.

In [ ]:
# Switch to the 0.6B (loads from the PVC cache), measure, then switch back.
from common import vllm_admin

small = "Qwen/Qwen3-0.6B"
vllm_admin.switch_model(small)
print(f"{small}: about {decode_rate(small):.0f} tokens/s")
# vllm_admin.switch_model(settings.model_name)   # switch back when you are done

# The MoE would go here, but it does not fit a 20 GB card. On a 24 GB+ card at INT4:
#   vllm_admin.switch_model is single-model; serve Qwen/Qwen3-30B-A3B on a larger GPU
#   and call decode_rate("Qwen/Qwen3-30B-A3B") to compare its per-token read.

## 6. What this means for owning your inference

The memory-bound limit is how you pick a model for a card. A small dense model and a sparse MoE can hold the same latency on the same GPU for different reasons: the dense model is small enough to read fast, and the MoE is large but reads only its active experts. Knowing which you have tells you what to expect and what to tune.

## 7. Hand off to Omer

Decode is memory-bound. There are five ways to read fewer bytes or reuse the bytes you read, and you just saw the first.

- MoE: read only the active experts (this module).
- Quantization: fewer bytes per weight (Module 5, Omer).
- Attention kernels and fusion: fewer trips to GPU memory during attention (Module 7, Omer).
- Speculative decoding: more accepted tokens per target-model step (Module 6, Omer).
- Continuous batching: one weight read shared across many requests, up to the knee (Module 7, Omer).

Each one points back to the plot you drew.

## Things to know

- **The limit is per operation.** Prefill sits near the compute line, decode far down the memory side, on the same card.
- **The ridge point is a property of your GPU.** Put an H100's numbers in and it moves. The shape of the lesson does not.
- **MoE trades footprint for bandwidth.** You store every expert, and you read only the active ones. You pay in VRAM to save on the per-token read.
- **Use the dense figure for the card peak.** The RTX 4000 Ada is about 107 TFLOP/s dense FP16. The 427 on the spec sheet is FP8 with 2:1 sparsity.

## Try it yourself

**Recompute the ridge.** Put an H100's numbers in section 2 (about 990 TFLOP/s dense FP16, 3.35 TB/s) and read where the crossover moves. **Stretch:** mark where decode at batch 16 sits, and watch it climb the memory line toward the ridge.

**Compare another MoE.** Find a different MoE's total and active parameter counts in its config, and compute its bytes per token against a dense model of the same total size.

In [ ]:
# Change these, then run the cell.
your_peak_tflops = 990.0    # e.g. an H100, dense FP16
your_bw_tbs = 3.35          # e.g. an H100, 3.35 TB/s

your_ridge = your_peak_tflops / your_bw_tbs
print(f"ridge point: {your_ridge:.0f} FLOPs/byte")

## Summary

- Decode is memory-bound: one token reads every weight once, so memory bandwidth caps the rate.
- The roofline plots compute against memory, and the ridge point is where they cross. Decode sits far down the memory side.
- A Mixture-of-Experts model reads only its active experts, so it moves fewer bytes per token than its total size and decodes faster.
- An MoE trades VRAM footprint for a smaller per-token read. It is the first of five ways to beat the memory limit.

## Next

**Hand off to Omer, Module 5: Quantization.** Decode is memory-bound, and quantization is the next way to read fewer bytes. Omer turns that into a decision framework: pick the right precision, understand the tradeoff, and measure what it costs in quality.